# Google Search Ranking & Discoverability Capstone

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/3bud-ZC/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring

This notebook is the capstone evidence layer. The gated warehouse model is built by `work/scripts/build_warehouse_capstone.py`; this notebook loads the committed receipts, verifies the final metrics, and presents the same evidence used by the deployed research paper.

## 1. Research question and decision

**Research question:** Which content pages with meaningful search visibility should a constrained SEO/content team review first because they are at elevated risk of a **>20% impression decline in the next calendar month**?

**Decision supported:** order a limited top-K human review queue.

The output is decision support only. A high score does not authorize an automatic rewrite, deletion, redirect, merge, or publication action.

In [ ]:
import os, json, subprocess
from pathlib import Path
import pandas as pd

REPO_URL = "https://github.com/3bud-ZC/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "work/outputs/capstone_warehouse_metrics.json").exists():
            return candidate
    return None

root = find_repo_root()
if root is None:
    if not Path(REPO_DIR).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    root = Path(REPO_DIR).resolve()

os.chdir(root)

metrics = json.loads(Path("work/outputs/capstone_warehouse_metrics.json").read_text(encoding="utf-8"))
top10 = json.loads(Path("work/outputs/capstone_recommendations_top10.json").read_text(encoding="utf-8"))

print("Question:", metrics["question"])
print("Lane:", metrics["lane"])
print("Grain:", metrics["grain"])
print("Selected model:", metrics["selected_model"])


## 2. Data

The final capstone uses the **real gated FlyRank warehouse release**, not the 30k starter slice.

- Dataset: `FlyRank/internship-warehouse`
- Build: `flyrank_pseudonymized_warehouse_release_v20260703`
- Documented daily fact rows: **78,835,655**
- Full documented daily date range: **2025-01-27 → 2026-06-30**
- Tables used: `dim_clients` and `fact_content_daily_performance`
- Final study queries **January–June 2026**
- Modeling grain: **one pseudonymized content item at one monthly decision point**

Eligibility requires meaningful recent visibility and sufficient client history. No client names, domains, raw URLs, private queries, credentials, or identifying exports are published.

In [ ]:
release = metrics["release"]
stats = pd.DataFrame(metrics["snapshot_stats"]).T
stats.index.name = "target_month"

print(json.dumps(release, indent=2))
display(stats.round(3))

assert release["daily_fact_documented_rows"] == 78_835_655
assert metrics["validation_design"]["sealed_final_test"] == "2026-06"


## 3. Target, features, baseline, and validation design

**Target:** the next calendar month's impressions are less than **80%** of the immediately preceding month's impressions.

**Features:** historical GSC measurements only:
- recent impressions and clicks
- recent CTR
- recent average position
- position change
- impression momentum
- recent and previous active search days

**Baseline:** a transparent fixed score combining negative momentum, worsening position, and review value from visibility.

**Models:** Logistic Regression and Random Forest.

**Time-aware split:**
- March + April 2026 snapshots → training
- May 2026 → model selection
- June 2026 → sealed final test

**Primary metric:** Precision@50 because the operational decision is a constrained review queue.

In [ ]:
print("Features:")
for feature in metrics["features"]:
    print("-", feature)

print("\nValidation design:")
print(json.dumps(metrics["validation_design"], indent=2))

print("\nLeakage checks:")
print(json.dumps(metrics["leakage_checks"], indent=2))

assert metrics["leakage_checks"]["future_target_fields_used_as_features"] is False
assert metrics["leakage_checks"]["hash_ids_used_as_features"] is False
assert metrics["leakage_checks"]["target_month_sealed_until_final_evaluation"] is True


## 4. Model selection and sealed-test results

The model is selected on **May**, not on the final test month.

On May validation:
- fixed rule Precision@50 = **0.460**
- Logistic Regression Precision@50 = **0.660**
- Random Forest Precision@50 = **0.860**

Random Forest is therefore selected before the June test is opened.

On the sealed **June 2026** test:
- fixed rule Precision@50 = **0.680**
- Random Forest Precision@50 = **0.920**
- June test base rate = **0.692**

This is predictive ranking evidence under a past→future design. It is not causal evidence that refreshing a page causes recovery.

In [ ]:
validation = pd.DataFrame(metrics["validation_metrics"]).T
sealed = pd.DataFrame(metrics["sealed_test_metrics"]).T

print("=== May model-selection validation ===")
display(validation[["base_rate","precision_at_20","precision_at_50","precision_at_100","average_precision","roc_auc"]].round(3))

print("=== Sealed June test ===")
display(sealed[["base_rate","precision_at_20","precision_at_50","precision_at_100","average_precision","roc_auc"]].round(3))

assert metrics["selected_model"] == "random_forest"
assert round(metrics["sealed_test_metrics"]["random_forest"]["precision_at_50"], 2) == 0.92


## 5. Ranked recommendations

The model score orders the queue; **reason codes explain feature-time evidence; a human makes the decision**.

Example public-safe reason codes include:
- `recent_impression_decline`
- `position_worsening`
- `high_visibility_low_ctr`
- `reduced_active_days`
- `high_visibility`

The output never exposes real URLs or client identities.

In [ ]:
recommendations = pd.DataFrame(top10)
cols = ["rank","content_hash_id","model_score","impressions_recent","position_recent","impression_momentum","reason_codes","action"]
display(recommendations[cols])

assert len(recommendations) == 10
assert recommendations["content_hash_id"].str.startswith("content_").all()


## 6. Limitations and honest framing

This project **can** claim measured predictive ranking performance on the specified warehouse snapshot and time-aware evaluation.

It **cannot** claim:
- that any feature is a Google ranking factor,
- that the model explains why a page declined,
- that refreshing a flagged page will cause recovery,
- that a score should trigger automatic content changes,
- or that one decline threshold is universally correct.

The unbalanced client panel, seasonality, SERP changes, indexing events, campaigns, consolidation, and measurement noise can all create false positives or false negatives.

In [ ]:
safe_claim = (
    "On the specified FlyRank warehouse snapshot and time-aware evaluation, "
    "the selected model achieved Precision@50 of "
    f"{metrics['sealed_test_metrics']['random_forest']['precision_at_50']:.3f} "
    "versus "
    f"{metrics['sealed_test_metrics']['fixed_rule']['precision_at_50']:.3f} "
    "for the transparent baseline on the sealed June test."
)
print(safe_claim)
print("Human review required:", metrics["human_review_required"])
print("Automatic editing:", metrics["automatic_editing"])

assert metrics["human_review_required"] is True
assert metrics["automatic_editing"] is False


## 7. Reproducibility

The public repository contains:
- all weekly assignment notebooks under `work/notebooks/`
- the final warehouse pipeline at `work/scripts/build_warehouse_capstone.py`
- committed small metric/recommendation receipts
- the final comparison figure
- the deployed-paper source under `docs/`
- `submission/paper_url.txt` containing exactly the public paper URL

The bulk ranked queue is intentionally not committed because `work/**/*.csv` is git-ignored by the project's leak guard.

In [ ]:
required = [
    "work/scripts/build_warehouse_capstone.py",
    "work/outputs/capstone_warehouse_metrics.json",
    "work/outputs/capstone_recommendations_top10.json",
    "work/figures/capstone_model_comparison.svg",
    "docs/index.html",
    "docs/capstone_model_comparison.svg",
    "submission/paper_url.txt",
    "work/demo_outline.md",
    "work/social_post.md",
    "work/employer_summary.md",
]

for path in required:
    assert Path(path).exists(), path
    print("OK", path)

paper_url = Path("submission/paper_url.txt").read_text(encoding="utf-8").strip()
print("\nDeployed paper:", paper_url)
assert paper_url == "https://flyrank-ml-paper-production.up.railway.app/"


## 8. Acknowledgments & data credit

**Built on the FlyRank ML Internship dataset** — [FlyRank AI](https://flyrank.ai).

The warehouse release is pseudonymized and public outputs preserve that boundary.

## Final self-check

- [x] Real gated FlyRank warehouse used
- [x] Future target defined after the feature window
- [x] Time-aware train / validation / sealed-test design used
- [x] Transparent baseline compared on the same final test
- [x] Leakage checks recorded
- [x] Ranked public-safe recommendations generated
- [x] Human review / no-auto-edit policy explicit
- [x] Public research paper generated and deployed
- [x] `submission/paper_url.txt` contains the deployed URL
- [x] Demo outline + social cut + employer summary committed
